In [ ]:
import numpy as np
from sympy import symbols, Eq, solve
import pandas as pd
import cv2
import os

In [7]:
t, s = symbols('t s')

cx = 320.5
fx = 554.38

IMAGE_FOLDER = 'ambulance pics folder/ambulance pics'  # folder where your images live

clicked_point = None

def mouse_click(event, x, y, flags, param):
    global clicked_point
    if event == cv2.EVENT_LBUTTONDOWN:
        clicked_point = (x, y)
        print(f"Clicked pixel: x={x}, y={y}")

def get_ambulance_pixel(image_path):
    global clicked_point
    clicked_point = None
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not load image: {image_path}")
        return None
    clone = img.copy()
    cv2.namedWindow("Click on ambulance")
    cv2.setMouseCallback("Click on ambulance", mouse_click)
    while True:
        cv2.imshow("Click on ambulance", clone)
        key = cv2.waitKey(1) & 0xFF
        if clicked_point is not None:
            cv2.circle(clone, clicked_point, 5, (0, 0, 255), -1)
            cv2.imshow("Click on ambulance", clone)
            cv2.waitKey(500)
            break
        if key == ord('q'):
            break
    cv2.destroyAllWindows()
    return clicked_point

def pixel_to_angle(x_pixel):
    theta = np.arctan((x_pixel - cx) / fx)
    print(f"theta from pixel: {np.degrees(theta):.2f} degrees")
    return theta

def midpoint(x1, y1, x2, y2):
    return np.array([(x1+x2)/2, (y1+y2)/2])

def distance(x1, y1, x2, y2):
    return np.sqrt((x2-x1)**2 + (y2-y1)**2)

def compute_p3(x1, y1, x2, y2, s1, d1, theta):
    chord_dx = x2 - x1
    chord_dy = y2 - y1
    u = np.array([chord_dx/d1, chord_dy/d1])
    n = np.array([-u[1], u[0]])
    offset = (d1/4) * np.abs(np.sin(theta))
    side = 1 if theta >= 0 else -1
    return s1 + side * offset * n

def new_perpendicular(x2, y2):
    return np.array([-y2, x2])

def find_center(x1, y1, s1, s2, n1, n2):
    ray_1 = s1 + n1*t
    ray_2 = s2 + n2*s
    solution = solve((Eq(ray_1[0], ray_2[0]), Eq(ray_1[1], ray_2[1])), (t, s))
    if isinstance(solution, list):
        if len(solution) == 0:
            return None
        solution = solution[0]
    s_variable = solution[s]
    center = s2 + n2*s_variable
    return float(center[0]), float(center[1])

def radius(x1, y1, h, k):
    return np.sqrt((x1-h)**2 + (y1-k)**2)

def final_vel(v, r, x1, y1, x2, y2, h, k):
    if r == 0:
        return 0
    cross = (x2-x1)*(k-y1) - (y2-y1)*(h-x1)
    sign = 1 if cross > 0 else -1
    return sign * v / r

def process_row(row, theta, v=0.5):
    x1, y1 = row['RobotX'], row['RobotY']
    x2, y2 = row['AmbX'],   row['AmbY']
    s1 = midpoint(x1, y1, x2, y2)
    d1 = distance(x1, y1, x2, y2)
    p3 = compute_p3(x1, y1, x2, y2, s1, d1, theta)
    s2 = midpoint(x1, y1, p3[0], p3[1])
    chord = np.array([x2-x1, y2-y1])
    n1 = new_perpendicular(chord[0], chord[1])
    n2 = new_perpendicular(p3[0]-x1, p3[1]-y1)
    center = find_center(x1, y1, s1, s2, n1, n2)
    if center is None:
        return None
    h, k = center
    r = radius(x1, y1, h, k)
    omega = final_vel(v, r, x1, y1, x2, y2, h, k)
    return omega

def build_image_path(image_number):
    path = os.path.join(IMAGE_FOLDER, f"image{int(image_number)}.png")
    if os.path.exists(path):
        return path
    return None

In [ ]:




df = pd.read_excel('distance_calculator.xlsx', header=1)

omegas = []

for idx, row in df.iterrows():
    print(f"\n--- Row {idx} | Image {int(row['Image #'])} ---")

    image_path = build_image_path(row['Image #'])
    if image_path is None:
        print(f"No image found for Image #{int(row['Image #'])}, skipping")
        omegas.append(None)
        continue

    pixel = get_ambulance_pixel(image_path)
    if pixel is None:
        print(f"No click registered, skipping")
        omegas.append(None)
        continue

    theta = pixel_to_angle(pixel[0])
    omega = process_row(row, theta)
    print(f"ω = {omega}")
    omegas.append(omega)

df['omega'] = omegas
df.to_csv('output_results.csv', index=False)


--- Row 0 | Image 1 ---
Clicked pixel: x=298, y=219
theta from pixel: -2.32 degrees
ω = 0.01923482062021058

--- Row 1 | Image 2 ---
Clicked pixel: x=270, y=215
theta from pixel: -5.20 degrees
ω = 0.06905692193694057

--- Row 2 | Image 3 ---
Clicked pixel: x=201, y=215
theta from pixel: -12.16 degrees
ω = 0.09357317267073603

--- Row 3 | Image 4 ---
Clicked pixel: x=213, y=216
theta from pixel: -10.97 degrees
ω = 0.08596152664521824

--- Row 4 | Image 5 ---
Clicked pixel: x=255, y=225
theta from pixel: -6.74 degrees
ω = 0.030187343936504844

--- Row 5 | Image 6 ---
Clicked pixel: x=323, y=233
theta from pixel: 0.26 degrees
ω = -0.0028390601782329323

--- Row 6 | Image 7 ---
Clicked pixel: x=295, y=234
theta from pixel: -2.63 degrees
ω = 0.030479350391646162

--- Row 7 | Image 8 ---
Clicked pixel: x=337, y=234
theta from pixel: 1.70 degrees
ω = -0.006985441333415586

--- Row 8 | Image 9 ---
Clicked pixel: x=371, y=237
theta from pixel: 5.20 degrees
ω = -0.011012731398876078

--- Row 9 

C:\Users\acrfa\AppData\Local\Temp\ipykernel_15948\3304859525.py:53: RuntimeWarning: invalid value encountered in scalar divide
  u = np.array([chord_dx/d1, chord_dy/d1])
C:\Users\acrfa\AppData\Local\Temp\ipykernel_15948\3304859525.py:64: RuntimeWarning: invalid value encountered in multiply
  ray_2 = s2 + n2*s
C:\Users\acrfa\AppData\Local\Temp\ipykernel_15948\3304859525.py:64: RuntimeWarning: invalid value encountered in add
  ray_2 = s2 + n2*s


Clicked pixel: x=592, y=236
theta from pixel: 26.09 degrees
ω = -0.04473134002318951

--- Row 176 | Image 177 ---
Clicked pixel: x=638, y=230
theta from pixel: 29.80 degrees
ω = -0.1040374180254759

--- Row 177 | Image 178 ---
Clicked pixel: x=206, y=236
theta from pixel: -11.67 degrees
ω = 0.013439492991075202

--- Row 178 | Image 179 ---
Clicked pixel: x=105, y=236
theta from pixel: -21.24 degrees
ω = 0.04560973821780777

--- Row 179 | Image 180 ---
Clicked pixel: x=128, y=235
theta from pixel: -19.15 degrees
ω = 0.022113903108285184

--- Row 180 | Image 181 ---
Clicked pixel: x=628, y=234
theta from pixel: 29.02 degrees
ω = -0.031787943992450086

--- Row 181 | Image 182 ---
Clicked pixel: x=53, y=236
theta from pixel: -25.76 degrees
ω = 0.02775434268668223

--- Row 182 | Image 183 ---
Clicked pixel: x=0, y=224
theta from pixel: -30.03 degrees
ω = 0.4565151059612467

--- Row 183 | Image 184 ---
Clicked pixel: x=8, y=212
theta from pixel: -29.41 degrees
ω = 0.2812725902411137

--- Row

In [6]:
import os
for f in os.listdir(IMAGE_FOLDER):
    print(f)

image1.png
image10.png
image100.png
image101.png
image102.png
image103.png
image104.png
image105.png
image106.png
image107.png
image108.png
image109.png
image11.png
image110.png
image111.png
image112.png
image113.png
image114.png
image115.png
image116.png
image117.png
image118.png
image119.png
image12.png
image120.png
image121.png
image122.png
image123.png
image124.png
image125.png
image126.png
image127.png
image128.png
image129.png
image13.png
image130.png
image131.png
image132.png
image133.png
image134.png
image135.png
image136.png
image137.png
image138.png
image139.png
image14.png
image140.png
image141.png
image142.png
image143.png
image144.png
image145.png
image146.png
image147.png
image148.png
image149.png
image15.png
image150.png
image151.png
image152.png
image153.png
image154.png
image155.png
image156.png
image157.png
image158.png
image159.png
image16.png
image160.png
image161.png
image162.png
image163.png
image164.png
image165.png
image166.png
image167.png
image168.png
image169

In [10]:
print(omegas)

[np.float64(0.01923482062021058), np.float64(0.06905692193694057), np.float64(0.09357317267073603), np.float64(0.08596152664521824), np.float64(0.030187343936504844), np.float64(-0.0028390601782329323), np.float64(0.030479350391646162), np.float64(-0.006985441333415586), np.float64(-0.011012731398876078), np.float64(-0.010391708667537245), np.float64(-0.051714409346894016), np.float64(-0.09483458669391726), np.float64(-0.06172705575898944), np.float64(-0.008804625357637822), np.float64(-0.017058168840812783), np.float64(0.045398963198054025), np.float64(-0.015304298647638447), np.float64(-0.019862755822903564), np.float64(-0.010186532620416092), np.float64(0.1028899552341801), np.float64(0.005317086303873123), np.float64(0.1560423523601992), np.float64(0.07902375222001642), np.float64(0.06784086584031707), np.float64(0.01825608940985618), np.float64(-0.012971825266102634), np.float64(0.0048504818518106795), np.float64(0.002401264705872229), np.float64(-0.0020201498166976532), np.float6